In [13]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.model_selection import GridSearchCV



from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix)

In [2]:
TH_500 = pd.read_csv(
    "../classification/TomsHardware/Relative_labeling/sigma=500/TomsHardware-Relative-Sigma-500.data",
    sep=",",
    header=None
)

TH_1000= pd.read_csv(
    "../classification/TomsHardware/Relative_labeling/sigma=1000/TomsHardware-Relative-Sigma-1000.data",
    sep=",",
    header=None
)
TH_1500= pd.read_csv(
    "../classification/TomsHardware/Relative_labeling/sigma=1500/TomsHardware-Relative-Sigma-1500.data",
    sep=",",
    header=None
)
groups = [
    "NCD", "BL", "NAD", "AI", "NAC", "ND",
    "CS", "AT", "NA", "ADL", "AS_NA", "AS_NAC"
]

columns = []
for group in groups:
    for t in range(8):
        columns.append(f"{group}_{t}")

columns.append("label")  

TH_500.columns = columns
TH_1000.columns = columns
TH_1500.columns = columns

prefixes = {col.split("_")[0] for col in TH_500.columns if "_" in col}
prefixes = {col.split("_")[0] for col in TH_1000.columns if "_" in col}
prefixes = {col.split("_")[0] for col in TH_1500.columns if "_" in col}

### 500

### 1.Baseline SVM without scaling

In [4]:
X = TH_500.drop("label", axis=1)
y = TH_500["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

svm_baseline = SVC(probability=True,random_state=42).fit(X_train, y_train)

y_pred_svm = svm_baseline.predict(X_test)
y_prob_svm = svm_baseline.predict_proba(X_test)[:, 1]

accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm)
recall_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)
roc_auc_svm = roc_auc_score(y_test, y_prob_svm)

cm_svm = confusion_matrix(y_test, y_pred_svm)


print('1.Baseline SVM without scaling')
print("Accuracy:", accuracy_svm)
print("Precision:", precision_svm)
print("Recall:", recall_svm)
print("F1-score:", f1_svm)
print("ROC-AUC:", roc_auc_svm)

print("\nConfusion Matrix:")
print(cm_svm)

1.Baseline SVM without scaling
Accuracy: 0.9184060721062619
Precision: 0.8498402555910544
Recall: 0.764367816091954
F1-score: 0.8048411497730711
ROC-AUC: 0.9652935089632799

Confusion Matrix:
[[1186   47]
 [  82  266]]


### 2.Baseline SVM with StandardScaler

In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


svm_scaled = SVC(probability=True, random_state=42).fit(X_train_scaled, y_train)

y_pred_svm_scaled = svm_scaled.predict(X_test_scaled)
y_prob_svm_scaled = svm_scaled.predict_proba(X_test_scaled)[:,1]


accuracy_svm_scaled = accuracy_score(y_test, y_pred_svm_scaled)
precision_svm_scaled = precision_score(y_test, y_pred_svm_scaled)
recall_svm_scaled = recall_score(y_test, y_pred_svm_scaled)
f1_svm_scaled = f1_score(y_test, y_pred_svm_scaled)
roc_auc_svm_scaled = roc_auc_score(y_test, y_prob_svm_scaled)

cm_svm_scaled = confusion_matrix(y_test, y_pred_svm_scaled)

print('2.Baseline SVM with StandardScaler')
print("Accuracy:", accuracy_svm_scaled)
print("Precision:", precision_svm_scaled)
print("Recall:", recall_svm_scaled)
print("F1-score:", f1_svm_scaled)
print("ROC-AUC:", roc_auc_svm_scaled)

print("\nConfusion Matrix:")
print(cm_svm_scaled)

2.Baseline SVM with StandardScaler
Accuracy: 0.900695762175838
Precision: 0.8398576512455516
Recall: 0.6781609195402298
F1-score: 0.7503974562798092
ROC-AUC: 0.9397227582478023

Confusion Matrix:
[[1188   45]
 [ 112  236]]


### 3.Balanced SVM with scaling

In [8]:
svm_balanced = SVC(probability=True,class_weight="balanced",random_state=42).fit(X_train_scaled, y_train)

y_pred_svm_bal = svm_balanced.predict(X_test_scaled)
y_prob_svm_bal = svm_balanced.predict_proba(X_test_scaled)[:,1]


accuracy_svm_bal = accuracy_score(y_test, y_pred_svm_bal)
precision_svm_bal = precision_score(y_test, y_pred_svm_bal)
recall_svm_bal = recall_score(y_test, y_pred_svm_bal)
f1_svm_bal = f1_score(y_test, y_pred_svm_bal)
roc_auc_svm_bal = roc_auc_score(y_test, y_prob_svm_bal)

cm_svm_bal = confusion_matrix(y_test, y_pred_svm_bal)

print('3.Balanced SVM with scaling')
print("Accuracy:", accuracy_svm_bal)
print("Precision:", precision_svm_bal)
print("Recall:", recall_svm_bal)
print("F1-score:", f1_svm_bal)
print("ROC-AUC:", roc_auc_svm_bal)

print("\nConfusion Matrix:")
print(cm_svm_bal)

3.Balanced SVM with scaling
Accuracy: 0.8779253636938646
Precision: 0.6773455377574371
Recall: 0.8505747126436781
F1-score: 0.7541401273885351
ROC-AUC: 0.9397950051738122

Confusion Matrix:
[[1092  141]
 [  52  296]]


### 4.SVM with Stratified K-Fold Cross-Validation

In [12]:
pipeline = Pipeline([("scaler", StandardScaler()),("svm", SVC(probability=True, random_state=42))])

cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results_svm = cross_validate(pipeline, X, y,cv=cv,scoring=scoring)


print('4.SVM with Stratified K-Fold Cross-Validation')
print("Accuracy:", cv_results_svm["test_accuracy"].mean())
print("Precision:", cv_results_svm["test_precision"].mean())
print("Recall:", cv_results_svm["test_recall"].mean())
print("F1-score:", cv_results_svm["test_f1"].mean())
print("ROC-AUC:", cv_results_svm["test_roc_auc"].mean())

4.SVM with Stratified K-Fold Cross-Validation
Accuracy: 0.8886780518659076
Precision: 0.8260590297692729
Recall: 0.6271844020683068
F1-score: 0.7121031703608363
ROC-AUC: 0.9402377183952332


### 5.SVM with Grid Search

In [15]:
pipeline = Pipeline([("scaler", StandardScaler()),("svm", SVC(probability=True, random_state=42))])

param_grid = {
    "svm__kernel": ["linear", "rbf"],
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale", "auto"]
}

grid_svm = GridSearchCV( pipeline, param_grid, cv=5, scoring="f1", n_jobs=-1).fit(X_train, y_train)

best_svm = grid_svm.best_estimator_


y_pred_svm_grid = best_svm.predict(X_test)
y_prob_svm_grid = best_svm.predict_proba(X_test)[:,1]

accuracy_svm_grid = accuracy_score(y_test, y_pred_svm_grid)
precision_svm_grid = precision_score(y_test, y_pred_svm_grid)
recall_svm_grid = recall_score(y_test, y_pred_svm_grid)
f1_svm_grid = f1_score(y_test, y_pred_svm_grid)
roc_auc_svm_grid = roc_auc_score(y_test, y_prob_svm_grid)

cm_svm_grid = confusion_matrix(y_test, y_pred_svm_grid)

print('SVM with Grid Search')
print("Best parameters:", grid_svm.best_params_)
print("Accuracy:", accuracy_svm_grid)
print("Precision:", precision_svm_grid)
print("Recall:", recall_svm_grid)
print("F1-score:", f1_svm_grid)
print("ROC-AUC:", roc_auc_svm_grid)

print("\nConfusion Matrix:")
print(cm_svm_grid)

SVM with Grid Search
Best parameters: {'svm__C': 10, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
Accuracy: 0.9032258064516129
Precision: 0.8115015974440895
Recall: 0.7298850574712644
F1-score: 0.7685325264750378
ROC-AUC: 0.9440995236363975

Confusion Matrix:
[[1174   59]
 [  94  254]]


### 1000

### 1.Baseline SVM without scaling

In [17]:
X = TH_1000.drop("label", axis=1)
y = TH_1000["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

svm_baseline = SVC(probability=True,random_state=42).fit(X_train, y_train)

y_pred_svm = svm_baseline.predict(X_test)
y_prob_svm = svm_baseline.predict_proba(X_test)[:, 1]

accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm)
recall_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)
roc_auc_svm = roc_auc_score(y_test, y_prob_svm)

cm_svm = confusion_matrix(y_test, y_pred_svm)


print('1.Baseline SVM without scaling')
print("Accuracy:", accuracy_svm)
print("Precision:", precision_svm)
print("Recall:", recall_svm)
print("F1-score:", f1_svm)
print("ROC-AUC:", roc_auc_svm)

print("\nConfusion Matrix:")
print(cm_svm)

1.Baseline SVM without scaling
Accuracy: 0.9424414927261227
Precision: 0.8393782383419689
Recall: 0.7297297297297297
F1-score: 0.7807228915662651
ROC-AUC: 0.9748440493473607

Confusion Matrix:
[[1328   31]
 [  60  162]]


### 2.Baseline SVM with StandardScaler

In [18]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


svm_scaled = SVC(probability=True, random_state=42).fit(X_train_scaled, y_train)

y_pred_svm_scaled = svm_scaled.predict(X_test_scaled)
y_prob_svm_scaled = svm_scaled.predict_proba(X_test_scaled)[:,1]


accuracy_svm_scaled = accuracy_score(y_test, y_pred_svm_scaled)
precision_svm_scaled = precision_score(y_test, y_pred_svm_scaled)
recall_svm_scaled = recall_score(y_test, y_pred_svm_scaled)
f1_svm_scaled = f1_score(y_test, y_pred_svm_scaled)
roc_auc_svm_scaled = roc_auc_score(y_test, y_prob_svm_scaled)

cm_svm_scaled = confusion_matrix(y_test, y_pred_svm_scaled)

print('2.Baseline SVM with StandardScaler')
print("Accuracy:", accuracy_svm_scaled)
print("Precision:", precision_svm_scaled)
print("Recall:", recall_svm_scaled)
print("F1-score:", f1_svm_scaled)
print("ROC-AUC:", roc_auc_svm_scaled)

print("\nConfusion Matrix:")
print(cm_svm_scaled)

2.Baseline SVM with StandardScaler
Accuracy: 0.9354838709677419
Precision: 0.8488372093023255
Recall: 0.6576576576576577
F1-score: 0.7411167512690355
ROC-AUC: 0.959966589105662

Confusion Matrix:
[[1333   26]
 [  76  146]]


### 3.Balanced SVM with scaling

In [19]:
svm_balanced = SVC(probability=True,class_weight="balanced",random_state=42).fit(X_train_scaled, y_train)

y_pred_svm_bal = svm_balanced.predict(X_test_scaled)
y_prob_svm_bal = svm_balanced.predict_proba(X_test_scaled)[:,1]


accuracy_svm_bal = accuracy_score(y_test, y_pred_svm_bal)
precision_svm_bal = precision_score(y_test, y_pred_svm_bal)
recall_svm_bal = recall_score(y_test, y_pred_svm_bal)
f1_svm_bal = f1_score(y_test, y_pred_svm_bal)
roc_auc_svm_bal = roc_auc_score(y_test, y_prob_svm_bal)

cm_svm_bal = confusion_matrix(y_test, y_pred_svm_bal)

print('3.Balanced SVM with scaling')
print("Accuracy:", accuracy_svm_bal)
print("Precision:", precision_svm_bal)
print("Recall:", recall_svm_bal)
print("F1-score:", f1_svm_bal)
print("ROC-AUC:", roc_auc_svm_bal)

print("\nConfusion Matrix:")
print(cm_svm_bal)

3.Balanced SVM with scaling
Accuracy: 0.9139784946236559
Precision: 0.6442953020134228
Recall: 0.8648648648648649
F1-score: 0.7384615384615385
ROC-AUC: 0.9604571458876096

Confusion Matrix:
[[1253  106]
 [  30  192]]


### 4.SVM with Stratified K-Fold Cross-Validation

In [20]:
pipeline = Pipeline([("scaler", StandardScaler()),("svm", SVC(probability=True, random_state=42))])

cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results_svm = cross_validate(pipeline, X, y,cv=cv,scoring=scoring)


print('4.SVM with Stratified K-Fold Cross-Validation')
print("Accuracy:", cv_results_svm["test_accuracy"].mean())
print("Precision:", cv_results_svm["test_precision"].mean())
print("Recall:", cv_results_svm["test_recall"].mean())
print("F1-score:", cv_results_svm["test_f1"].mean())
print("ROC-AUC:", cv_results_svm["test_roc_auc"].mean())

4.SVM with Stratified K-Fold Cross-Validation
Accuracy: 0.932953826691967
Precision: 0.8388267385612173
Recall: 0.6486486486486487
F1-score: 0.7306381656762011
ROC-AUC: 0.9642390735105966


### 5.SVM with Grid Search

In [21]:
pipeline = Pipeline([("scaler", StandardScaler()),("svm", SVC(probability=True, random_state=42))])

param_grid = {
    "svm__kernel": ["linear", "rbf"],
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale", "auto"]
}

grid_svm = GridSearchCV( pipeline, param_grid, cv=5, scoring="f1", n_jobs=-1).fit(X_train, y_train)

best_svm = grid_svm.best_estimator_


y_pred_svm_grid = best_svm.predict(X_test)
y_prob_svm_grid = best_svm.predict_proba(X_test)[:,1]

accuracy_svm_grid = accuracy_score(y_test, y_pred_svm_grid)
precision_svm_grid = precision_score(y_test, y_pred_svm_grid)
recall_svm_grid = recall_score(y_test, y_pred_svm_grid)
f1_svm_grid = f1_score(y_test, y_pred_svm_grid)
roc_auc_svm_grid = roc_auc_score(y_test, y_prob_svm_grid)

cm_svm_grid = confusion_matrix(y_test, y_pred_svm_grid)

print('SVM with Grid Search')
print("Best parameters:", grid_svm.best_params_)
print("Accuracy:", accuracy_svm_grid)
print("Precision:", precision_svm_grid)
print("Recall:", recall_svm_grid)
print("F1-score:", f1_svm_grid)
print("ROC-AUC:", roc_auc_svm_grid)

print("\nConfusion Matrix:")
print(cm_svm_grid)

SVM with Grid Search
Best parameters: {'svm__C': 10, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
Accuracy: 0.9348513598987982
Precision: 0.8051282051282052
Recall: 0.7072072072072072
F1-score: 0.7529976019184652
ROC-AUC: 0.9579049247923421

Confusion Matrix:
[[1321   38]
 [  65  157]]


### 1500

### 1.Baseline SVM without scaling

In [22]:
X = TH_1500.drop("label", axis=1)
y = TH_1500["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

svm_baseline = SVC(probability=True,random_state=42).fit(X_train, y_train)

y_pred_svm = svm_baseline.predict(X_test)
y_prob_svm = svm_baseline.predict_proba(X_test)[:, 1]

accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm)
recall_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)
roc_auc_svm = roc_auc_score(y_test, y_prob_svm)

cm_svm = confusion_matrix(y_test, y_pred_svm)


print('1.Baseline SVM without scaling')
print("Accuracy:", accuracy_svm)
print("Precision:", precision_svm)
print("Recall:", recall_svm)
print("F1-score:", f1_svm)
print("ROC-AUC:", roc_auc_svm)

print("\nConfusion Matrix:")
print(cm_svm)

1.Baseline SVM without scaling
Accuracy: 0.958886780518659
Precision: 0.8601398601398601
Recall: 0.7321428571428571
F1-score: 0.7909967845659164
ROC-AUC: 0.9868609510329256

Confusion Matrix:
[[1393   20]
 [  45  123]]


### 2.Baseline SVM with StandardScaler

In [23]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


svm_scaled = SVC(probability=True, random_state=42).fit(X_train_scaled, y_train)

y_pred_svm_scaled = svm_scaled.predict(X_test_scaled)
y_prob_svm_scaled = svm_scaled.predict_proba(X_test_scaled)[:,1]


accuracy_svm_scaled = accuracy_score(y_test, y_pred_svm_scaled)
precision_svm_scaled = precision_score(y_test, y_pred_svm_scaled)
recall_svm_scaled = recall_score(y_test, y_pred_svm_scaled)
f1_svm_scaled = f1_score(y_test, y_pred_svm_scaled)
roc_auc_svm_scaled = roc_auc_score(y_test, y_prob_svm_scaled)

cm_svm_scaled = confusion_matrix(y_test, y_pred_svm_scaled)

print('2.Baseline SVM with StandardScaler')
print("Accuracy:", accuracy_svm_scaled)
print("Precision:", precision_svm_scaled)
print("Recall:", recall_svm_scaled)
print("F1-score:", f1_svm_scaled)
print("ROC-AUC:", roc_auc_svm_scaled)

print("\nConfusion Matrix:")
print(cm_svm_scaled)

2.Baseline SVM with StandardScaler
Accuracy: 0.9538266919671095
Precision: 0.8861788617886179
Recall: 0.6488095238095238
F1-score: 0.7491408934707904
ROC-AUC: 0.974446466484683

Confusion Matrix:
[[1399   14]
 [  59  109]]


### 3.Balanced SVM with scaling

In [24]:
svm_balanced = SVC(probability=True,class_weight="balanced",random_state=42).fit(X_train_scaled, y_train)

y_pred_svm_bal = svm_balanced.predict(X_test_scaled)
y_prob_svm_bal = svm_balanced.predict_proba(X_test_scaled)[:,1]


accuracy_svm_bal = accuracy_score(y_test, y_pred_svm_bal)
precision_svm_bal = precision_score(y_test, y_pred_svm_bal)
recall_svm_bal = recall_score(y_test, y_pred_svm_bal)
f1_svm_bal = f1_score(y_test, y_pred_svm_bal)
roc_auc_svm_bal = roc_auc_score(y_test, y_prob_svm_bal)

cm_svm_bal = confusion_matrix(y_test, y_pred_svm_bal)

print('3.Balanced SVM with scaling')
print("Accuracy:", accuracy_svm_bal)
print("Precision:", precision_svm_bal)
print("Recall:", recall_svm_bal)
print("F1-score:", f1_svm_bal)
print("ROC-AUC:", roc_auc_svm_bal)

print("\nConfusion Matrix:")
print(cm_svm_bal)

3.Balanced SVM with scaling
Accuracy: 0.9424414927261227
Precision: 0.6681222707423581
Recall: 0.9107142857142857
F1-score: 0.7707808564231738
ROC-AUC: 0.9809886091733225

Confusion Matrix:
[[1337   76]
 [  15  153]]


### 4.SVM with Stratified K-Fold Cross-Validation

In [25]:
pipeline = Pipeline([("scaler", StandardScaler()),("svm", SVC(probability=True, random_state=42))])

cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results_svm = cross_validate(pipeline, X, y,cv=cv,scoring=scoring)


print('4.SVM with Stratified K-Fold Cross-Validation')
print("Accuracy:", cv_results_svm["test_accuracy"].mean())
print("Precision:", cv_results_svm["test_precision"].mean())
print("Recall:", cv_results_svm["test_recall"].mean())
print("F1-score:", cv_results_svm["test_f1"].mean())
print("ROC-AUC:", cv_results_svm["test_roc_auc"].mean())

4.SVM with Stratified K-Fold Cross-Validation
Accuracy: 0.9558507273877292
Precision: 0.8793156800992653
Recall: 0.6777542969850663
F1-score: 0.7650824772373067
ROC-AUC: 0.9697001316958623


### 5.SVM with Grid Search

In [26]:
pipeline = Pipeline([("scaler", StandardScaler()),("svm", SVC(probability=True, random_state=42))])

param_grid = {
    "svm__kernel": ["linear", "rbf"],
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale", "auto"]
}

grid_svm = GridSearchCV( pipeline, param_grid, cv=5, scoring="f1", n_jobs=-1).fit(X_train, y_train)

best_svm = grid_svm.best_estimator_


y_pred_svm_grid = best_svm.predict(X_test)
y_prob_svm_grid = best_svm.predict_proba(X_test)[:,1]

accuracy_svm_grid = accuracy_score(y_test, y_pred_svm_grid)
precision_svm_grid = precision_score(y_test, y_pred_svm_grid)
recall_svm_grid = recall_score(y_test, y_pred_svm_grid)
f1_svm_grid = f1_score(y_test, y_pred_svm_grid)
roc_auc_svm_grid = roc_auc_score(y_test, y_prob_svm_grid)

cm_svm_grid = confusion_matrix(y_test, y_pred_svm_grid)

print('SVM with Grid Search')
print("Best parameters:", grid_svm.best_params_)
print("Accuracy:", accuracy_svm_grid)
print("Precision:", precision_svm_grid)
print("Recall:", recall_svm_grid)
print("F1-score:", f1_svm_grid)
print("ROC-AUC:", roc_auc_svm_grid)

print("\nConfusion Matrix:")
print(cm_svm_grid)

SVM with Grid Search
Best parameters: {'svm__C': 10, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
Accuracy: 0.9576217583807717
Precision: 0.8796992481203008
Recall: 0.6964285714285714
F1-score: 0.7774086378737541
ROC-AUC: 0.9767591750075826

Confusion Matrix:
[[1397   16]
 [  51  117]]


### Result

Výsledky ukazujú, že algoritmus SVM vykazuje veľmi vysokú schopnosť separácie tried pre všetky tri datasety, čo potvrdzujú vysoké hodnoty ROC-AUC (0,94–0,99).
Najlepšie výsledky vo všetkých prípadoch boli dosiahnuté pre základnú konfiguráciu SVM bez škálovania príznakov, kde F1-score dosiahol 0,805 pre σ = 500.
Aplikácia škálovania príznakov, vyváženia tried a optimalizácie hyperparametrov menila pomer medzi Precision a Recall, avšak neviedla k výraznému zlepšeniu F1-score v porovnaní so základným modelom.
Taktiež sa pozoruje tendencia rastu Accuracy a ROC-AUC so zvyšovaním hodnoty σ, čo naznačuje zlepšenie separovateľnosti tried v datasetoch s vyšším prahom.

In [1]:
import pandas as pd

columns = ["Model", "Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]

data_500 = [
    ["SVM (no scaling)", 0.9184, 0.8498, 0.7644, 0.8048, 0.9653],
    ["SVM (StandardScaler)", 0.9007, 0.8399, 0.6782, 0.7504, 0.9397],
    ["Balanced SVM", 0.8779, 0.6773, 0.8506, 0.7541, 0.9398],
    ["SVM Stratified K-Fold", 0.8887, 0.8261, 0.6272, 0.7121, 0.9402],
    ["SVM Grid Search", 0.9032, 0.8115, 0.7299, 0.7685, 0.9441],
]

df_500 = pd.DataFrame(data_500, columns=columns)
df_500

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,SVM (no scaling),0.9184,0.8498,0.7644,0.8048,0.9653
1,SVM (StandardScaler),0.9007,0.8399,0.6782,0.7504,0.9397
2,Balanced SVM,0.8779,0.6773,0.8506,0.7541,0.9398
3,SVM Stratified K-Fold,0.8887,0.8261,0.6272,0.7121,0.9402
4,SVM Grid Search,0.9032,0.8115,0.7299,0.7685,0.9441


In [2]:
data_1000 = [
    ["SVM (no scaling)", 0.9424, 0.8394, 0.7297, 0.7807, 0.9748],
    ["SVM (StandardScaler)", 0.9355, 0.8488, 0.6577, 0.7411, 0.9600],
    ["Balanced SVM", 0.9140, 0.6443, 0.8649, 0.7385, 0.9605],
    ["SVM Stratified K-Fold", 0.9330, 0.8388, 0.6486, 0.7306, 0.9642],
    ["SVM Grid Search", 0.9349, 0.8051, 0.7072, 0.7530, 0.9579],
]

df_1000 = pd.DataFrame(data_1000, columns=columns)
df_1000

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,SVM (no scaling),0.9424,0.8394,0.7297,0.7807,0.9748
1,SVM (StandardScaler),0.9355,0.8488,0.6577,0.7411,0.9600
2,Balanced SVM,0.9140,0.6443,0.8649,0.7385,0.9605
3,SVM Stratified K-Fold,0.9330,0.8388,0.6486,0.7306,0.9642
4,SVM Grid Search,0.9349,0.8051,0.7072,0.7530,0.9579
